# MT response and a measured QC decision

**Level 1 — Foundations**  
**Prerequisites:** Python basics; 35 min.

## Objectives

Trace the evidence from source contract to result. State units and assumptions, reproduce the calculation, and identify which inference the evidence cannot support.

## Setup

Uses repository modules and committed records through a discovered root; it downloads and mutates nothing. Install exact pins: `pip install numpy==2.2.6 matplotlib==3.11.2` `scipy==1.15.2` is required by the MT forward operator.

## Data

Measured QC-only record plus a separate seeded synthetic calibration artifact; never merge these evidence classes.

## Method

Measured Clear Lake `cl061` contains 42 frequencies and complex off-diagonal impedances; its subsurface truth is unknown. The product’s tensor-compatibility screen fails, so the receipt marks it QC-only and ineligible for 1D inversion. That is evidence of model inadequacy, not permission to tune a fit until it looks plausible. With `exp(+iωt)`, impedance `Z=E/H` is in Ω, apparent resistivity `ρₐ=|Z|²/(μ₀ω)` in Ω·m, and phase `φ=atan2(Im Z,Re Z)` in degrees; `T=1/f` is seconds. Plot the measured XY/YX responses against period and discuss sign conventions, dimensionality, galvanic distortion and why QC precedes inversion.

Parameters are log-resistivities: positive by construction, with equal steps representing equal multiplicative changes. Complex residuals use per-real-component sigma; real and imaginary parts both inform the fit. Frequencies are not independent depth samples: sensitivity kernels overlap, so one period does not identify one interface. Local sensitivity and coverage remain conditional on the fixed parameterization and cannot rule out global equivalence.

## Results

Read the physical axes and units; distinguish data fit, error against synthetic truth and agreement with analyst labels. Recomputed values must agree with the serialized receipt. A plotted model alone does not establish uniqueness, field transfer or resolution.

### Diagnostic 1

Inspect the committed case record and provenance.

In [ ]:
from pathlib import Path
from importlib.metadata import version
print("Runtime versions:", {name: version(name) for name in ['numpy', 'matplotlib', 'scipy']})
import numpy as np
root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/"data").is_dir() and (p/"data-pipeline").is_dir())
# Inspect measured cl061 QC receipt; no inversion is eligible.
import json
from pathlib import Path
case=json.loads((root/"data/derived/v2/edi/clear-lake-cl061-screen.json").read_text(encoding="utf-8")); f=np.asarray(case["frequencies_hz"])
print(case["id"],case["source_kind"],"N=",len(f),"truth=",case["truth"],"eligible=",case["one_d_inversion_eligible"],"compatibility=",case["compatibility"])

### Diagnostic 2

Compute the declared comparison and plot it with physical units.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# Plot measured XY/YX curves on period (s) axes.
import matplotlib.pyplot as plt
period=1/f; fig,ax=plt.subplots(2,1,figsize=(8,6),sharex=True,constrained_layout=True)
for n,c in [("xy","#007c91"),("yx","#bf5b17")]:
 q=case["observed"][n]; ax[0].loglog(period,q["apparent"],".-",label="Z"+n,color=c); ax[1].semilogx(period,q["phase"],".-",label="Z"+n,color=c)
ax[0].set_ylabel("Apparent resistivity (Ω·m)"); ax[1].set_ylabel("Phase (degrees)"); ax[1].set_xlabel("Period (s)")
for a in ax: a.grid(True,which="both",alpha=.25); a.legend()
ax[0].set_title("Measured Clear Lake cl061 · QC only"); plt.show()

## Summary

Conclusion is conditional on geometry, acquisition, noise model, split, parameterization and source rights. Preserve negative cases and name the next observation or control needed.

## References

- [https://github.com/MTgeophysics/mtpy-v2](https://github.com/MTgeophysics/mtpy-v2)
- [https://www.usgs.gov/data/magnetotelluric-data-clear-lake-region-northern-california](https://www.usgs.gov/data/magnetotelluric-data-clear-lake-region-northern-california)